# MMIR Paper Plots

All plots for the paper submission. Each section is titled for easy identification.

Plots are saved to: `C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Plots`

In [ ]:
# ============================================================
# Common Imports & Configuration
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.lines import Line2D

# Output directory
OUT_DIR = r"C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# Font Settings (Times New Roman / STIX for paper)
# ============================================================
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Liberation Serif", "STIXGeneral"],
    "mathtext.fontset": "stix",
    "mathtext.rm": "STIXGeneral",
    "mathtext.it": "STIXGeneral:italic",
    "mathtext.bf": "STIXGeneral:bold",
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14
})

# ============================================================
# Shared label formatter
# ============================================================
def pretty_label(name):
    name = name.lower()
    if "bm25_text" in name:  return r"$\mathrm{BM25}_\mathrm{Text}$"
    if "openclip_image" in name: return r"$\mathrm{OpenCLIP}_\mathrm{Image}$"
    if "clip_image" in name: return r"$\mathrm{CLIP}_\mathrm{Image}$"
    if "clip_text" in name:  return r"$\mathrm{CLIP}_\mathrm{Text}$"
    if "flava_image" in name: return r"$\mathrm{FLAVA}_\mathrm{Image}$"
    if "uniir_image" in name: return r"$\mathrm{UniIR}_\mathrm{Image}$"
    if "uniir_joint" in name: return r"$\mathrm{UniIR}_\mathrm{Joint}$"
    if "llama-nc" in name:   return r"$\mathrm{llama\text{-}nc}$"
    if "blip2" in name:      return r"$\mathrm{BLIP2}$"
    if "qwen" in name:       return r"$\mathrm{Qwen3\text{-}VL}$"
    return name

# ============================================================
# Shared heatmap plotting function
# ============================================================
def plot_heatmap(matrix, metric_label, save_name, title=None):
    """Plot and save a heatmap with the paper style."""
    plt.figure(figsize=(14, 6))
    ax = sns.heatmap(
        matrix.astype(float),
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        center=0,
        linewidths=0.6,
        linecolor="black",
        mask=matrix.isna(),
        vmin=-20,
        vmax=40,
        annot_kws={"fontsize": 15},
        cbar_kws={"label": f"{metric_label} (%) Improvement"}
    )

    ax.set_xlabel("Reranker", fontsize=22, fontweight="bold", labelpad=15)
    ax.set_ylabel("Retriever", fontsize=22, fontweight="bold", labelpad=15)

    ax.set_xticklabels([pretty_label(c) for c in matrix.columns], rotation=40, ha="right", fontsize=18)
    ax.set_yticklabels([pretty_label(r) for r in matrix.index], rotation=0, fontsize=18)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_color("black")

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label(f"{metric_label} (%) Improvement", fontsize=18)

    if title:
        plt.title(title, fontsize=20, fontweight="bold", pad=15)

    plt.tight_layout()
    png_path = os.path.join(OUT_DIR, f"{save_name}.png")
    pdf_path = os.path.join(OUT_DIR, f"{save_name}.pdf")
    plt.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.savefig(pdf_path, dpi=600, bbox_inches="tight", format="pdf")
    plt.show()
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

# ============================================================
# Shared dumbbell plotting function
# ============================================================
def plot_dumbbell(data_list, ylim_min, save_name):
    """Dumbbell plot: recall at k=10, 30, 100 for R@1, R@5, R@10."""
    df = pd.DataFrame(data_list)
    models = df['model'].unique()
    metrics = ['R@1', 'R@5', 'R@10']
    colors = {'R@1': '#264653', 'R@5': '#e76f51', 'R@10': '#2a9d8f'}
    offsets = {'R@1': -0.2, 'R@5': 0, 'R@10': 0.2}
    x = np.arange(len(models))

    fig, ax = plt.subplots(figsize=(16, 9), dpi=120)

    # Vertical separators between models
    for i in range(len(models) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='-', alpha=0.15, linewidth=3)

    for metric in metrics:
        for i, model in enumerate(models):
            vals = df[(df.model == model) & (df.metric == metric)]
            if vals.empty:
                continue

            y10 = vals[vals.k == 10]['value'].values[0] if not vals[vals.k == 10].empty else None
            y30 = vals[vals.k == 30]['value'].values[0] if not vals[vals.k == 30].empty else None
            y100 = vals[vals.k == 100]['value'].values[0] if not vals[vals.k == 100].empty else None
            xpos = x[i] + offsets[metric]

            y_points = [y for y in [y10, y30, y100] if y is not None]
            if not y_points:
                continue
            ymin, ymax = min(y_points), max(y_points)

            # Connector line
            ax.vlines(xpos, ymin, ymax, color=colors[metric], alpha=0.4, linewidth=4.0, zorder=1)

            # Points: circle=k10, triangle=k30, square=k100
            if y10 is not None:
                ax.scatter(xpos, y10, color=colors[metric], s=130, marker='o',
                           edgecolor='white', linewidth=2.0, zorder=3)
            if y30 is not None:
                ax.scatter(xpos, y30, color=colors[metric], s=150, marker='^',
                           edgecolor='white', linewidth=2.0, zorder=3)
            if y100 is not None:
                ax.scatter(xpos, y100, color=colors[metric], s=150, marker='s',
                           edgecolor='white', linewidth=2.0, zorder=3)

    ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=0.15)
    ax.xaxis.grid(False)

    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_color('black')
        ax.spines[spine].set_linewidth(1.5)

    # Format x-tick labels with subscript
    formatted_labels = []
    for model in models:
        parts = model.split('-')
        label = f"$\\mathrm{{{parts[0]}}}_{{ \\mathrm{{{parts[1]}}} }}$"
        formatted_labels.append(label)

    plt.xticks(x, formatted_labels, fontsize=28, fontweight='medium', color='#333')
    plt.yticks(fontsize=30, color='black')
    plt.ylabel("Recall Score", fontsize=36, labelpad=15, fontweight='bold', color='#333')
    plt.ylim(ylim_min, 1.0)

    legend_handles = [
        Line2D([0], [0], color=colors['R@1'], marker='o', linestyle='',
               markeredgecolor='white', markersize=12, label='R@1'),
        Line2D([0], [0], color=colors['R@5'], marker='o', linestyle='',
               markeredgecolor='white', markersize=12, label='R@5'),
        Line2D([0], [0], color=colors['R@10'], marker='o', linestyle='',
               markeredgecolor='white', markersize=12, label='R@10'),
        Line2D([0], [0], color='none', label=''),
        Line2D([0], [0], color='gray', marker='o', linestyle='',
               markeredgecolor='white', markersize=12, label='k=10'),
        Line2D([0], [0], color='gray', marker='^', linestyle='',
               markeredgecolor='white', markersize=12, label='k=30'),
        Line2D([0], [0], color='gray', marker='s', linestyle='',
               markeredgecolor='white', markersize=12, label='k=100'),
    ]

    ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0, 1), ncol=2,
              frameon=True, fancybox=True, facecolor='white', edgecolor='black', fontsize=20)

    plt.tight_layout()
    png_path = os.path.join(OUT_DIR, f"{save_name}.png")
    pdf_path = os.path.join(OUT_DIR, f"{save_name}.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.show()
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

print("Common setup loaded.")

---
## 1. Reranking R@1 Improvement Heatmap — COCO

Heatmap showing R@1 (%) improvement when using each **Retriever** (row) with each **Reranker** (column).

Includes BM25 as retriever. Excludes ColPali reranker.

In [ ]:
# ============================================================
# 1. Reranking R@1 Improvement Heatmap — COCO
# ============================================================

# Row order (Retriever)
rows_coco = [
    "bm25_text",
    "clip_text",
    "clip_image",
    "flava_image",
    "uniir_image",
    "uniir_joint-image-text",
    "llama-nc",
    "openclip_image"
]

# Column order (Reranker)
cols_coco = [
    "clip_image",
    "flava_image",
    "openclip_image",
    "uniir_image",
    "uniir_joint-image-text",
    "llama-nc",
    "qwen",
    "blip2"
]

# Initialize matrix with NaN
coco_matrix = pd.DataFrame(np.nan, index=rows_coco, columns=cols_coco)

# --- Hardcoded R@1 improvement values (from existing heatmap) ---

# BM25_Text (retriever)
coco_matrix.loc["bm25_text", "clip_image"]              = 45.00
coco_matrix.loc["bm25_text", "flava_image"]              = 56.12
coco_matrix.loc["bm25_text", "uniir_image"]              = 81.12
coco_matrix.loc["bm25_text", "uniir_joint-image-text"]   = 93.58
coco_matrix.loc["bm25_text", "blip2"]                    = 79.55

# CLIP_Text (retriever)
coco_matrix.loc["clip_text", "clip_image"]               = 1.54
coco_matrix.loc["clip_text", "flava_image"]              = 7.57
coco_matrix.loc["clip_text", "openclip_image"]           = 22.83
coco_matrix.loc["clip_text", "uniir_image"]              = 30.93
coco_matrix.loc["clip_text", "uniir_joint-image-text"]   = 39.73
coco_matrix.loc["clip_text", "qwen"]                     = 51.57
coco_matrix.loc["clip_text", "blip2"]                    = 45.29

# CLIP_Image (retriever)
coco_matrix.loc["clip_image", "clip_image"]              = 0.00
coco_matrix.loc["clip_image", "flava_image"]             = 7.97
coco_matrix.loc["clip_image", "openclip_image"]          = 26.13
coco_matrix.loc["clip_image", "uniir_image"]             = 34.52
coco_matrix.loc["clip_image", "uniir_joint-image-text"]  = 46.04
coco_matrix.loc["clip_image", "llama-nc"]                = 47.90
coco_matrix.loc["clip_image", "qwen"]                    = 49.76
coco_matrix.loc["clip_image", "blip2"]                   = 72.49

# FLAVA_Image (retriever)
coco_matrix.loc["flava_image", "clip_image"]             = -1.27
coco_matrix.loc["flava_image", "flava_image"]            = 0.00
coco_matrix.loc["flava_image", "openclip_image"]         = 21.22
coco_matrix.loc["flava_image", "uniir_image"]            = 29.85
coco_matrix.loc["flava_image", "uniir_joint-image-text"] = 41.62
coco_matrix.loc["flava_image", "llama-nc"]               = 42.18
coco_matrix.loc["flava_image", "qwen"]                   = 43.67
coco_matrix.loc["flava_image", "blip2"]                  = 65.16

# UniIR_Image (retriever)
coco_matrix.loc["uniir_image", "clip_image"]             = -24.68
coco_matrix.loc["uniir_image", "flava_image"]            = -20.03
coco_matrix.loc["uniir_image", "openclip_image"]         = -6.53
coco_matrix.loc["uniir_image", "uniir_image"]            = 0.00
coco_matrix.loc["uniir_image", "uniir_joint-image-text"] = 8.85
coco_matrix.loc["uniir_image", "llama-nc"]               = 9.84
coco_matrix.loc["uniir_image", "qwen"]                   = 11.21
coco_matrix.loc["uniir_image", "blip2"]                  = 27.71

# UniIR_Joint (retriever)
coco_matrix.loc["uniir_joint-image-text", "clip_image"]              = -28.84
coco_matrix.loc["uniir_joint-image-text", "flava_image"]             = -24.97
coco_matrix.loc["uniir_joint-image-text", "openclip_image"]          = -13.48
coco_matrix.loc["uniir_joint-image-text", "uniir_image"]             = -8.10
coco_matrix.loc["uniir_joint-image-text", "uniir_joint-image-text"]  = 0.00
coco_matrix.loc["uniir_joint-image-text", "llama-nc"]                = 2.64
coco_matrix.loc["uniir_joint-image-text", "qwen"]                    = 10.30
coco_matrix.loc["uniir_joint-image-text", "blip2"]                   = 17.42

# llama-nc (retriever)
coco_matrix.loc["llama-nc", "llama-nc"]                  = 0.00
coco_matrix.loc["llama-nc", "blip2"]                     = 17.53

# OpenCLIP_Image (retriever)
coco_matrix.loc["openclip_image", "openclip_image"]      = 0.00
coco_matrix.loc["openclip_image", "qwen"]                = 19.39
coco_matrix.loc["openclip_image", "blip2"]               = 37.16

# Plot
plot_heatmap(coco_matrix, "R@1", "coco_heatmap_R1_improvement")

---
## 2. Reranking R@1 Improvement Heatmap — Flickr30K

Heatmap showing R@1 (%) improvement when using each **Retriever** (row) with each **Reranker** (column).

Includes BM25 as retriever. Excludes ColPali reranker.

In [ ]:
# ============================================================
# 2. Reranking R@1 Improvement Heatmap — Flickr30K
# ============================================================

# Row order (Retriever)
rows_flickr = [
    "bm25_text",
    "clip_text",
    "clip_image",
    "flava_image",
    "uniir_image",
    "uniir_joint-image-text",
    "llama-nc",
    "openclip_image"
]

# Column order (Reranker)
cols_flickr = [
    "clip_image",
    "flava_image",
    "openclip_image",
    "uniir_image",
    "uniir_joint-image-text",
    "llama-nc",
    "qwen",
    "blip2"
]

# Initialize matrix with NaN
flickr_matrix = pd.DataFrame(np.nan, index=rows_flickr, columns=cols_flickr)

# --- Hardcoded R@1 improvement values (from existing heatmap) ---

# BM25_Text (retriever)
flickr_matrix.loc["bm25_text", "clip_image"]              = 37.88
flickr_matrix.loc["bm25_text", "flava_image"]             = 35.33
flickr_matrix.loc["bm25_text", "uniir_image"]             = 54.04
flickr_matrix.loc["bm25_text", "uniir_joint-image-text"]  = 62.82
flickr_matrix.loc["bm25_text", "blip2"]                   = 53.58

# CLIP_Text (retriever)
flickr_matrix.loc["clip_text", "clip_image"]              = 9.23
flickr_matrix.loc["clip_text", "flava_image"]             = 7.01
flickr_matrix.loc["clip_text", "openclip_image"]          = 22.32
flickr_matrix.loc["clip_text", "uniir_image"]             = 19.93
flickr_matrix.loc["clip_text", "uniir_joint-image-text"]  = 32.29
flickr_matrix.loc["clip_text", "qwen"]                    = 38.01
flickr_matrix.loc["clip_text", "blip2"]                   = 33.03

# CLIP_Image (retriever)
flickr_matrix.loc["clip_image", "clip_image"]             = 0.00
flickr_matrix.loc["clip_image", "flava_image"]            = -3.68
flickr_matrix.loc["clip_image", "openclip_image"]         = 16.29
flickr_matrix.loc["clip_image", "uniir_image"]            = 17.51
flickr_matrix.loc["clip_image", "uniir_joint-image-text"] = 28.55
flickr_matrix.loc["clip_image", "llama-nc"]               = 25.39
flickr_matrix.loc["clip_image", "qwen"]                   = 31.35
flickr_matrix.loc["clip_image", "blip2"]                  = 33.45

# FLAVA_Image (retriever)
flickr_matrix.loc["flava_image", "clip_image"]            = 7.65
flickr_matrix.loc["flava_image", "flava_image"]           = 0.00
flickr_matrix.loc["flava_image", "openclip_image"]        = 23.51
flickr_matrix.loc["flava_image", "uniir_image"]           = 25.93
flickr_matrix.loc["flava_image", "uniir_joint-image-text"]= 37.13
flickr_matrix.loc["flava_image", "llama-nc"]              = 33.02
flickr_matrix.loc["flava_image", "qwen"]                  = 38.99
flickr_matrix.loc["flava_image", "blip2"]                 = 42.16

# UniIR_Image (retriever)
flickr_matrix.loc["uniir_image", "clip_image"]            = -14.48
flickr_matrix.loc["uniir_image", "flava_image"]           = -18.21
flickr_matrix.loc["uniir_image", "openclip_image"]        = -0.75
flickr_matrix.loc["uniir_image", "uniir_image"]           = 0.00
flickr_matrix.loc["uniir_image", "uniir_joint-image-text"]= 9.40
flickr_matrix.loc["uniir_image", "llama-nc"]              = 7.01
flickr_matrix.loc["uniir_image", "qwen"]                  = 12.39
flickr_matrix.loc["uniir_image", "blip2"]                 = 13.88

# UniIR_Joint (retriever)
flickr_matrix.loc["uniir_joint-image-text", "clip_image"]             = -20.08
flickr_matrix.loc["uniir_joint-image-text", "flava_image"]            = -22.13
flickr_matrix.loc["uniir_joint-image-text", "openclip_image"]         = -8.61
flickr_matrix.loc["uniir_joint-image-text", "uniir_image"]            = -8.33
flickr_matrix.loc["uniir_joint-image-text", "uniir_joint-image-text"] = 0.00
flickr_matrix.loc["uniir_joint-image-text", "llama-nc"]               = -1.37
flickr_matrix.loc["uniir_joint-image-text", "qwen"]                   = 6.01
flickr_matrix.loc["uniir_joint-image-text", "blip2"]                  = 5.19

# llama-nc (retriever)
flickr_matrix.loc["llama-nc", "llama-nc"]                 = 0.00
flickr_matrix.loc["llama-nc", "blip2"]                    = 4.42

# OpenCLIP_Image (retriever)
flickr_matrix.loc["openclip_image", "openclip_image"]     = 0.00
flickr_matrix.loc["openclip_image", "qwen"]               = 13.25
flickr_matrix.loc["openclip_image", "blip2"]              = 15.06

# Plot
plot_heatmap(flickr_matrix, "R@1", "flickr_heatmap_R1_improvement")

---
## 3. BLIP2 Reranking Recall at k=10, 30, 100 — MSCOCO

Dumbbell plot showing R@1, R@5, R@10 recall scores for each retriever model reranked by BLIP2,
across candidate pool sizes k=10, k=30, k=100.

Models: CLIP-Image, CLIP-Text, FLAVA-Image, UniIR-Image, UniIR-Joint, OpenCLIP-Image

(FLAVA-Text and UniIR-Text excluded)

In [ ]:
# ============================================================
# 3. BLIP2 Reranking Recall at k=10, 30, 100 — MSCOCO
# ============================================================

# All hardcoded values (from model_recalls_report.json)
mscoco_rerank_data = [
    # CLIP-Image
    {"model":"CLIP-Image","metric":"R@1","k":10,"value":0.5888},
    {"model":"CLIP-Image","metric":"R@1","k":30,"value":0.63},
    {"model":"CLIP-Image","metric":"R@1","k":100,"value":0.6504},
    {"model":"CLIP-Image","metric":"R@5","k":10,"value":0.7146},
    {"model":"CLIP-Image","metric":"R@5","k":30,"value":0.81},
    {"model":"CLIP-Image","metric":"R@5","k":100,"value":0.8536},
    {"model":"CLIP-Image","metric":"R@10","k":10,"value":0.7238},
    {"model":"CLIP-Image","metric":"R@10","k":30,"value":0.84},
    {"model":"CLIP-Image","metric":"R@10","k":100,"value":0.9048},
    # CLIP-Text
    {"model":"CLIP-Text","metric":"R@1","k":10,"value":0.5646},
    {"model":"CLIP-Text","metric":"R@1","k":30,"value":0.61},
    {"model":"CLIP-Text","metric":"R@1","k":100,"value":0.6392},
    {"model":"CLIP-Text","metric":"R@5","k":10,"value":0.6736},
    {"model":"CLIP-Text","metric":"R@5","k":30,"value":0.74},
    {"model":"CLIP-Text","metric":"R@5","k":100,"value":0.7634},
    {"model":"CLIP-Text","metric":"R@10","k":10,"value":0.6916},
    {"model":"CLIP-Text","metric":"R@10","k":30,"value":0.79},
    {"model":"CLIP-Text","metric":"R@10","k":100,"value":0.8258},
    # FLAVA-Image
    {"model":"FLAVA-Image","metric":"R@1","k":10,"value":0.6218},
    {"model":"FLAVA-Image","metric":"R@1","k":30,"value":0.65},
    {"model":"FLAVA-Image","metric":"R@1","k":100,"value":0.6492},
    {"model":"FLAVA-Image","metric":"R@5","k":10,"value":0.7848},
    {"model":"FLAVA-Image","metric":"R@5","k":30,"value":0.85},
    {"model":"FLAVA-Image","metric":"R@5","k":100,"value":0.8568},
    {"model":"FLAVA-Image","metric":"R@10","k":10,"value":0.7998},
    {"model":"FLAVA-Image","metric":"R@10","k":30,"value":0.90},
    {"model":"FLAVA-Image","metric":"R@10","k":100,"value":0.913},
    # UniIR-Image
    {"model":"UniIR-Image","metric":"R@1","k":10,"value":0.6374},
    {"model":"UniIR-Image","metric":"R@1","k":30,"value":0.65},
    {"model":"UniIR-Image","metric":"R@1","k":100,"value":0.6498},
    {"model":"UniIR-Image","metric":"R@5","k":10,"value":0.8258},
    {"model":"UniIR-Image","metric":"R@5","k":30,"value":0.85},
    {"model":"UniIR-Image","metric":"R@5","k":100,"value":0.8596},
    {"model":"UniIR-Image","metric":"R@10","k":10,"value":0.8578},
    {"model":"UniIR-Image","metric":"R@10","k":30,"value":0.90},
    {"model":"UniIR-Image","metric":"R@10","k":100,"value":0.915},
    # UniIR-Joint
    {"model":"UniIR-Joint","metric":"R@1","k":10,"value":0.6456},
    {"model":"UniIR-Joint","metric":"R@1","k":30,"value":0.65},
    {"model":"UniIR-Joint","metric":"R@1","k":100,"value":0.6502},
    {"model":"UniIR-Joint","metric":"R@5","k":10,"value":0.7834},
    {"model":"UniIR-Joint","metric":"R@5","k":30,"value":0.77},
    {"model":"UniIR-Joint","metric":"R@5","k":100,"value":0.763},
    {"model":"UniIR-Joint","metric":"R@10","k":10,"value":0.8384},
    {"model":"UniIR-Joint","metric":"R@10","k":30,"value":0.84},
    {"model":"UniIR-Joint","metric":"R@10","k":100,"value":0.824},
    # OpenCLIP-Image
    {"model":"OpenCLIP-Image","metric":"R@1","k":10,"value":0.6288},
    {"model":"OpenCLIP-Image","metric":"R@1","k":30,"value":0.6464},
    {"model":"OpenCLIP-Image","metric":"R@1","k":100,"value":0.6504},
    {"model":"OpenCLIP-Image","metric":"R@5","k":10,"value":0.7934},
    {"model":"OpenCLIP-Image","metric":"R@5","k":30,"value":0.8406},
    {"model":"OpenCLIP-Image","metric":"R@5","k":100,"value":0.8572},
    {"model":"OpenCLIP-Image","metric":"R@10","k":10,"value":0.8108},
    {"model":"OpenCLIP-Image","metric":"R@10","k":30,"value":0.8878},
    {"model":"OpenCLIP-Image","metric":"R@10","k":100,"value":0.9122},
]

# Plot MSCOCO
plot_dumbbell(mscoco_rerank_data, 0.5, 'mscoco_10_30_100_rerank')

---
## 4. BLIP2 Reranking Recall at k=10, 30, 100 — Flickr30K

Dumbbell plot showing R@1, R@5, R@10 recall scores for each retriever model reranked by BLIP2,
across candidate pool sizes k=10, k=30, k=100.

Models: CLIP-Image, CLIP-Text, FLAVA-Image, UniIR-Image, UniIR-Joint, OpenCLIP-Image

(FLAVA-Text and UniIR-Text excluded)

In [ ]:
# ============================================================
# 4. BLIP2 Reranking Recall at k=10, 30, 100 — Flickr30K
# ============================================================

flickr_rerank_data = [
    # CLIP-Image
    {"model":"CLIP-Image","metric":"R@1","k":10,"value":0.7520},
    {"model":"CLIP-Image","metric":"R@1","k":30,"value":0.76},
    {"model":"CLIP-Image","metric":"R@1","k":100,"value":0.762},
    {"model":"CLIP-Image","metric":"R@5","k":10,"value":0.8910},
    {"model":"CLIP-Image","metric":"R@5","k":30,"value":0.93},
    {"model":"CLIP-Image","metric":"R@5","k":100,"value":0.932},
    {"model":"CLIP-Image","metric":"R@10","k":10,"value":0.8980},
    {"model":"CLIP-Image","metric":"R@10","k":30,"value":0.95},
    {"model":"CLIP-Image","metric":"R@10","k":100,"value":0.962},
    # CLIP-Text
    {"model":"CLIP-Text","metric":"R@1","k":10,"value":0.7210},
    {"model":"CLIP-Text","metric":"R@1","k":30,"value":0.75},
    {"model":"CLIP-Text","metric":"R@1","k":100,"value":0.762},
    {"model":"CLIP-Text","metric":"R@5","k":10,"value":0.8110},
    {"model":"CLIP-Text","metric":"R@5","k":30,"value":0.86},
    {"model":"CLIP-Text","metric":"R@5","k":100,"value":0.867},
    {"model":"CLIP-Text","metric":"R@10","k":10,"value":0.8160},
    {"model":"CLIP-Text","metric":"R@10","k":30,"value":0.88},
    {"model":"CLIP-Text","metric":"R@10","k":100,"value":0.913},
    # FLAVA-Image
    {"model":"FLAVA-Image","metric":"R@1","k":10,"value":0.7480},
    {"model":"FLAVA-Image","metric":"R@1","k":30,"value":0.76},
    {"model":"FLAVA-Image","metric":"R@1","k":100,"value":0.762},
    {"model":"FLAVA-Image","metric":"R@5","k":10,"value":0.8680},
    {"model":"FLAVA-Image","metric":"R@5","k":30,"value":0.92},
    {"model":"FLAVA-Image","metric":"R@5","k":100,"value":0.929},
    {"model":"FLAVA-Image","metric":"R@10","k":10,"value":0.8780},
    {"model":"FLAVA-Image","metric":"R@10","k":30,"value":0.94},
    {"model":"FLAVA-Image","metric":"R@10","k":100,"value":0.960},
    # UniIR-Image
    {"model":"UniIR-Image","metric":"R@1","k":10,"value":0.7680},
    {"model":"UniIR-Image","metric":"R@1","k":30,"value":0.77},
    {"model":"UniIR-Image","metric":"R@1","k":100,"value":0.763},
    {"model":"UniIR-Image","metric":"R@5","k":10,"value":0.9230},
    {"model":"UniIR-Image","metric":"R@5","k":30,"value":0.95},
    {"model":"UniIR-Image","metric":"R@5","k":100,"value":0.934},
    {"model":"UniIR-Image","metric":"R@10","k":10,"value":0.9360},
    {"model":"UniIR-Image","metric":"R@10","k":30,"value":0.97},
    {"model":"UniIR-Image","metric":"R@10","k":100,"value":0.967},
    # UniIR-Joint
    {"model":"UniIR-Joint","metric":"R@1","k":10,"value":0.7780},
    {"model":"UniIR-Joint","metric":"R@1","k":30,"value":0.78},
    {"model":"UniIR-Joint","metric":"R@1","k":100,"value":0.77},
    {"model":"UniIR-Joint","metric":"R@5","k":10,"value":0.9050},
    {"model":"UniIR-Joint","metric":"R@5","k":30,"value":0.89},
    {"model":"UniIR-Joint","metric":"R@5","k":100,"value":0.864},
    {"model":"UniIR-Joint","metric":"R@10","k":10,"value":0.9290},
    {"model":"UniIR-Joint","metric":"R@10","k":30,"value":0.94},
    {"model":"UniIR-Joint","metric":"R@10","k":100,"value":0.914},
    # OpenCLIP-Image
    {"model":"OpenCLIP-Image","metric":"R@1","k":10,"value":0.763},
    {"model":"OpenCLIP-Image","metric":"R@1","k":30,"value":0.764},
    {"model":"OpenCLIP-Image","metric":"R@1","k":100,"value":0.764},
    {"model":"OpenCLIP-Image","metric":"R@5","k":10,"value":0.912},
    {"model":"OpenCLIP-Image","metric":"R@5","k":30,"value":0.938},
    {"model":"OpenCLIP-Image","metric":"R@5","k":100,"value":0.930},
    {"model":"OpenCLIP-Image","metric":"R@10","k":10,"value":0.923},
    {"model":"OpenCLIP-Image","metric":"R@10","k":30,"value":0.965},
    {"model":"OpenCLIP-Image","metric":"R@10","k":100,"value":0.964},
]

# Plot Flickr30K
plot_dumbbell(flickr_rerank_data, 0.7, 'flickr_10_30_100_rerank')

---
## 5. Pareto Frontier — Indexing Method Trade-offs

Bubble chart showing Latency vs Candidate Fidelity (Accuracy vs KNN %) for four indexing methods:
Exact KNN (Flat), HNSW, O-IVFPQ, IVFPQ.

Bubble size encodes storage footprint (GB). Pareto frontier line connects non-dominated points.

X-axis is log-scale latency.

In [ ]:
# ============================================================
# 5. Pareto Frontier — Indexing Method Trade-offs
# ============================================================

# Hardcoded data from Table 5
pareto_data = {
    'Algorithm': ['O-IVFPQ', 'HNSW', 'IVFPQ', 'Exact KNN'],
    'Latency_ms': [54, 96, 151, 6450],
    'Accuracy': [83.77, 90.73, 53.22, 100.00],
    'Storage_GB': [2.15, 22.89, 1.02, 21.25],
    'Color': ['#EC4899', '#3B82F6', '#8B5CF6', '#10B981'],
}

df_pareto = pd.DataFrame(pareto_data)

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(28, 19))

# Map bubble size to storage (GB)
base_size = 180 * 4
df_pareto['bubble_size'] = base_size + (df_pareto['Storage_GB'] * 45 * 4)

# 1. Plot the Pareto Frontier Line
frontier_x = [54, 96, 6450]
frontier_y = [83.77, 90.73, 100.00]
ax.plot(frontier_x, frontier_y, color='#E11D48', linestyle='--', linewidth=6.0, alpha=0.8,
        label='Pareto Frontier', zorder=2)

# 2. Draw dominated region shading
fill_x = [30] + frontier_x + [12000]
fill_y = [83.77, 83.77, 90.73, 100.00, 100.00]
ax.fill_between(fill_x, fill_y, 45, color='#475569', alpha=0.04, zorder=1, label='Trade-off Region')

# 3. Plot the scatter points
for idx, row in df_pareto.iterrows():
    ax.scatter(
        row['Latency_ms'], row['Accuracy'],
        s=row['bubble_size'],
        color=row['Color'],
        marker='o',
        alpha=0.9,
        edgecolor='black',
        linewidth=3.6,
        label=row['Algorithm'],
        zorder=3
    )

ax.set_xscale('log')
ax.set_xlim(20, 22000)
ax.set_ylim(48, 105)

# Ticks
x_ticks = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
x_labels = ['50 ms', '100 ms', '200 ms', '500 ms', '1.0 s', '2.0 s', '5.0 s', '10 s']
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, fontsize=40)

y_ticks = [50, 60, 70, 80, 90, 100]
y_labels = ['50%', '60%', '70%', '80%', '90%', '100%']
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=40)
ax.tick_params(axis='both', labelsize=40, width=3.0, length=12)

ax.set_xlabel('Latency (log scale)', fontsize=50, fontweight='bold', labelpad=25)
ax.set_ylabel('Candidate Fidelity (Acc vs KNN %)', fontsize=50, fontweight='bold', labelpad=25)

# Annotations
bbox_props = dict(boxstyle="round,pad=0.5", fc="white", ec="#D1D5DB", lw=1.2, alpha=0.95)

# O-IVFPQ
ax.annotate(
    "O-IVFPQ\nLatency: 54 ms\nAccuracy: 83.77%\nStorage: 2.15 GB",
    xy=(54, 83.77),
    xytext=(100, -120),
    textcoords='offset points',
    fontsize=28,
    fontweight='bold',
    bbox=bbox_props,
    arrowprops=dict(arrowstyle="->", color="#4B5563", lw=2.5, connectionstyle="arc3,rad=-0.1"),
    zorder=4
)

# HNSW
ax.annotate(
    "HNSW\nLatency: 96 ms\nAccuracy: 90.73%\nStorage: 22.89 GB",
    xy=(96, 90.73),
    xytext=(-100, 100),
    textcoords='offset points',
    fontsize=28,
    fontweight='bold',
    bbox=bbox_props,
    arrowprops=dict(arrowstyle="->", color="#4B5563", lw=2.5, connectionstyle="arc3,rad=0.1"),
    zorder=4
)

# IVFPQ
ax.annotate(
    "IVFPQ\nLatency: 151 ms\nAccuracy: 53.22%\nStorage: 1.02 GB",
    xy=(151, 53.22),
    xytext=(100, 80),
    textcoords='offset points',
    fontsize=28,
    fontweight='bold',
    bbox=bbox_props,
    arrowprops=dict(arrowstyle="->", color="#4B5563", lw=2.5, connectionstyle="arc3,rad=0.1"),
    zorder=4
)

# Exact KNN
ax.annotate(
    "Exact KNN (Flat Index)\nLatency: 6.45 s\nAccuracy: 100.00%\nStorage: 21.25 GB",
    xy=(6450, 100.00),
    xytext=(60, -160),
    textcoords='offset points',
    fontsize=28,
    fontweight='bold',
    bbox=bbox_props,
    arrowprops=dict(arrowstyle="->", color="#4B5563", lw=2.5, connectionstyle="arc3,rad=-0.1"),
    zorder=4
)

# Legend
h_header1 = Line2D([0], [0], color='none', label='')
h_header2 = Line2D([0], [0], color='none', label='')
h_header3 = Line2D([0], [0], color='none', label='')

h_knn = Line2D([0], [0], marker='o', color='none', markerfacecolor='#10B981',
               markeredgecolor='black', markeredgewidth=2.0, markersize=14)
h_hnsw = Line2D([0], [0], marker='o', color='none', markerfacecolor='#3B82F6',
                markeredgecolor='black', markeredgewidth=2.0, markersize=14)
h_oivfpq = Line2D([0], [0], marker='o', color='none', markerfacecolor='#EC4899',
                  markeredgecolor='black', markeredgewidth=2.0, markersize=14)
h_ivfpq = Line2D([0], [0], marker='o', color='none', markerfacecolor='#8B5CF6',
                 markeredgecolor='black', markeredgewidth=2.0, markersize=14)
h_frontier = Line2D([0], [0], color='#E11D48', linestyle='--', linewidth=3.5)
h_region = mpatches.Patch(color='#475569', alpha=0.18)

h_s1 = Line2D([0], [0], marker='o', color='none', markerfacecolor='#9CA3AF',
               markeredgecolor='black', markeredgewidth=1.5, markersize=8)
h_s5 = Line2D([0], [0], marker='o', color='none', markerfacecolor='#9CA3AF',
               markeredgecolor='black', markeredgewidth=1.5, markersize=12)
h_s15 = Line2D([0], [0], marker='o', color='none', markerfacecolor='#9CA3AF',
                markeredgecolor='black', markeredgewidth=1.5, markersize=18)
h_s25 = Line2D([0], [0], marker='o', color='none', markerfacecolor='#9CA3AF',
                markeredgecolor='black', markeredgewidth=1.5, markersize=24)

h_blank = Line2D([0], [0], color='none')

# 3-column legend inside plot
col1_h = [h_header1, h_knn, h_hnsw, h_oivfpq, h_ivfpq]
col1_l = ['INDEXING METHOD', 'Exact KNN (Flat)', 'HNSW', 'O-IVFPQ', 'IVFPQ']
col2_h = [h_header2, h_frontier, h_region, h_blank, h_blank]
col2_l = ['TRADE-OFFS', 'Pareto Frontier', 'Trade-off Region', '', '']
col3_h = [h_header3, h_s1, h_s5, h_s15, h_s25]
col3_l = ['STORAGE LEVEL', '1.0 GB', '5.0 GB', '15.0 GB', '25.0 GB']

legend_handles = col1_h + col2_h + col3_h
legend_labels = col1_l + col2_l + col3_l

legend = ax.legend(
    legend_handles, legend_labels,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.03),
    ncol=3,
    columnspacing=1.5,
    labelspacing=0.4,
    handletextpad=0.6,
    borderpad=0.6,
    frameon=True,
    shadow=False,
    fancybox=True,
    fontsize=21
)

# Style headers
for text in legend.get_texts():
    val = text.get_text()
    if val in ['INDEXING METHOD', 'TRADE-OFFS', 'STORAGE LEVEL']:
        text.set_fontweight('bold')
        text.set_fontsize(20)
        text.set_color('#374151')

frame = legend.get_frame()
frame.set_facecolor('white')
frame.set_edgecolor('#D1D5DB')
frame.set_linewidth(1.8)
frame.set_alpha(0.95)

ax.grid(axis='both', which='major', linestyle='-', alpha=0.5)
ax.grid(axis='both', which='minor', linestyle=':', alpha=0.2)

plt.tight_layout()

png_path = os.path.join(OUT_DIR, 'pareto_variant_A.png')
pdf_path = os.path.join(OUT_DIR, 'pareto_variant_A.pdf')
plt.savefig(png_path, dpi=300, bbox_inches='tight', pad_inches=0.25)
plt.savefig(pdf_path, dpi=300, bbox_inches='tight', pad_inches=0.25, format='pdf')
plt.show()
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")

---
## 6. Accuracy Scaling Dashboard — HNSW / IVFPQ / O-IVFPQ

3-panel figure showing Candidate Fidelity vs search parameter for each index family:
- **(a) HNSW**: Candidate Fidelity vs efSearch, multiple (M, efC) configs
- **(b) IVFPQ**: Candidate Fidelity vs nprobe, standard + high-capacity nlist configs
- **(c) O-IVFPQ**: Candidate Fidelity vs nprobe, OPQ-IVF configs

Best configuration per panel is marked with a star.

Data loaded from `master_overlap_accuracy.csv`.

In [ ]:
# ============================================================
# 6. Accuracy Scaling Dashboard — HNSW / IVFPQ / O-IVFPQ
# ============================================================

csv_path = r"C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Data\master_overlap_accuracy.csv"
df_master = pd.read_csv(csv_path)

# ---- Data Prep: HNSW ----
df_hnsw = df_master[df_master['index_type'] == 'hnsw'].copy()
df_hnsw.loc[df_hnsw['search_setting'] == 'root', 'search_param_value'] = 128.0
df_hnsw['search_param_value'] = df_hnsw['search_param_value'].astype(int)
df_hnsw = df_hnsw.sort_values(by=['method_file', 'search_param_value'])

def format_hnsw_name(filename):
    parts = filename.replace('.json', '').split('_')
    m_val, efc_val = '', ''
    for part in parts:
        if part.startswith('m') and part[1:].isdigit():
            m_val = part[1:]
        elif part.startswith('efc') and part[3:].isdigit():
            efc_val = part[3:]
    return f"HNSW (M={m_val}, efC={efc_val})"

df_hnsw['config_label'] = df_hnsw['method_file'].apply(format_hnsw_name)
hnsw_configs = sorted(df_hnsw['config_label'].unique())
hnsw_palette = sns.color_palette('tab10', n_colors=len(hnsw_configs))
hnsw_colors = {cfg: hnsw_palette[i % len(hnsw_palette)] for i, cfg in enumerate(hnsw_configs)}
hnsw_markers = ['o', 's', 'D', '^', 'v', 'p', 'd', '*', 'X']

# ---- Data Prep: IVFPQ (Standard + High-Capacity Plain) ----
df_ivfpq = df_master[df_master['index_type'] == 'ivfpq'].copy()
df_ivfpq['search_param_value'] = pd.to_numeric(df_ivfpq['search_param_value'], errors='coerce')
df_ivfpq.loc[df_ivfpq['search_setting'] == 'root', 'search_param_value'] = df_ivfpq.loc[
    df_ivfpq['search_setting'] == 'root', 'search_param_value'].fillna(8.0)
df_ivfpq['search_param_value'] = df_ivfpq['search_param_value'].astype(int)
df_ivfpq = df_ivfpq.sort_values(by=['method_file', 'search_param_value'])

def format_ivfpq_name(filename):
    parts = filename.replace('.json', '').split('_')
    if 'fac' in parts:
        opq_dim, ivf_val, pq_val = '', '', ''
        for part in parts:
            if part.startswith('opq'):  opq_dim = part[3:]
            elif part.startswith('ivf'): ivf_val = part[3:]
            elif part.startswith('pq'):  pq_val = part[2:]
        try:
            ivf_num = int(ivf_val)
            ivf_str = f"{ivf_num // 1024}k" if ivf_num >= 1024 else str(ivf_num)
        except: ivf_str = ivf_val
        return f"OPQ-IVF{ivf_str} (M={pq_val})"
    else:
        nlist_val, m_val = '', ''
        for part in parts:
            if part.startswith('nlist'):  nlist_val = part[5:]
            elif part.startswith('m') and part[1:].isdigit(): m_val = part[1:]
        try:
            nlist_num = int(nlist_val)
            nlist_str = f"{nlist_num // 1024}k" if nlist_num >= 1024 else str(nlist_num)
        except: nlist_str = nlist_val
        return f"IVFPQ-nlist{nlist_str} (M={m_val})"

df_ivfpq['config_label'] = df_ivfpq['method_file'].apply(format_ivfpq_name)

# Exclude OPQ configs from plain IVFPQ
df_ivfpq_nonopq = df_ivfpq[~df_ivfpq['method_file'].str.contains('opq|fac', case=False, na=False)].copy()

# Standard IVFPQ configs (excluding nlist16k and 32k)
df_ivfpq_std = df_ivfpq_nonopq[~df_ivfpq_nonopq['config_label'].str.contains('nlist16k|nlist32k')].copy()
std_configs = sorted(df_ivfpq_std['config_label'].unique())
std_palette = sns.color_palette('tab10', n_colors=max(3, len(std_configs)))
std_colors = {cfg: std_palette[i % len(std_palette)] for i, cfg in enumerate(std_configs)}

# Complete plain IVFPQ list (including 16k and 32k)
plain_configs = [
    'IVFPQ-nlist2k (M=16)',
    'IVFPQ-nlist4k (M=16)',
    'IVFPQ-nlist4k (M=32)',
    'IVFPQ-nlist4k (M=64)',
    'IVFPQ-nlist8k (M=16)',
    'IVFPQ-nlist8k (M=32)',
    'IVFPQ-nlist8k (M=64)',
    'IVFPQ-nlist16k (M=96)',
    'IVFPQ-nlist32k (M=96)'
]
df_ivfpq_plain_dashboard = df_ivfpq_nonopq[df_ivfpq_nonopq['config_label'].isin(plain_configs)].copy()
ivfpq_markers = ['o', 's', 'D', '^', 'v', 'p', 'd', '*', 'X']

# ---- Data Prep: OIVFPQ ----
df_oivfpq = df_ivfpq[df_ivfpq['method_file'].str.contains('opq|fac', case=False)].copy()
large_configs = sorted(df_oivfpq['config_label'].unique())
large_palette = sns.color_palette('Dark2', n_colors=max(3, len(large_configs)))
large_colors = {cfg: large_palette[i % len(large_palette)] for i, cfg in enumerate(large_configs)}

# ---- Generate 3-Panel Dashboard ----
sns.set_theme(style='whitegrid')

DASH_LABEL_FS = 30
DASH_TITLE_FS = 33

fig, axes = plt.subplots(1, 3, figsize=(28, 10))

# Panel 1: HNSW
for i, config in enumerate(hnsw_configs):
    subset = df_hnsw[df_hnsw['config_label'] == config]
    axes[0].plot(subset['search_param_value'], subset['mean_accuracy'],
                 marker=hnsw_markers[i % len(hnsw_markers)],
                 markersize=9, linewidth=2.2,
                 color=hnsw_colors.get(config, '#6B7280'),
                 label=config)
axes[0].set_xscale('log', base=2)
axes[0].set_xticks([128, 256, 512, 1024, 2048, 4096])
axes[0].set_xticklabels(['128', '256', '512', '1024', '2048', '4096'], fontsize=16)
axes[0].tick_params(axis='y', labelsize=16)
axes[0].set_xlabel('efSearch Parameter', fontsize=DASH_LABEL_FS, fontweight='bold', labelpad=8)
axes[0].set_ylabel('Candidate Fidelity', fontsize=DASH_LABEL_FS, fontweight='bold', labelpad=8)
axes[0].set_title('(a) HNSW', fontsize=DASH_TITLE_FS, fontweight='bold', pad=12)
axes[0].legend(loc='lower right', fontsize=11, frameon=True)

if not df_hnsw.empty:
    best_h = df_hnsw.loc[df_hnsw['mean_accuracy'].idxmax()]
    axes[0].scatter([best_h['search_param_value']], [best_h['mean_accuracy']],
                    s=220, marker='*', color='black', zorder=6)
    axes[0].annotate(f"Best: {best_h['mean_accuracy']*100:.2f}%",
                     (best_h['search_param_value'], best_h['mean_accuracy']),
                     xytext=(8,8), textcoords='offset points', fontsize=12)

# Panel 2: IVFPQ (Standard + High-Capacity)
for i, config in enumerate(std_configs):
    subset = df_ivfpq_plain_dashboard[df_ivfpq_plain_dashboard['config_label'] == config]
    axes[1].plot(subset['search_param_value'], subset['mean_accuracy'],
                 marker=ivfpq_markers[i % len(ivfpq_markers)],
                 markersize=9, linewidth=2.2,
                 color=std_colors[config],
                 label=config)

# Plot new high-capacity curves
new_configs = ['IVFPQ-nlist16k (M=96)', 'IVFPQ-nlist32k (M=96)']
new_colors = {
    'IVFPQ-nlist16k (M=96)': '#4F46E5',  # Indigo
    'IVFPQ-nlist32k (M=96)': '#EC4899',  # Pink
}
new_markers = {
    'IVFPQ-nlist16k (M=96)': '*',
    'IVFPQ-nlist32k (M=96)': 'X',
}
for config in new_configs:
    subset = df_ivfpq_plain_dashboard[df_ivfpq_plain_dashboard['config_label'] == config]
    axes[1].plot(subset['search_param_value'], subset['mean_accuracy'],
                 marker=new_markers[config],
                 markersize=10, linewidth=2.2,
                 color=new_colors[config],
                 label=config)

axes[1].set_xscale('log', base=2)
axes[1].set_xticks([8, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
axes[1].set_xticklabels(['8', '32', '64', '128', '256', '512', '1k', '2k', '4k', '8k', '16k', '32k'], fontsize=16)
axes[1].tick_params(axis='y', labelsize=16)
axes[1].set_xlabel('nprobe Parameter', fontsize=DASH_LABEL_FS, fontweight='bold', labelpad=8)
axes[1].set_ylabel('Candidate Fidelity', fontsize=DASH_LABEL_FS, fontweight='bold', labelpad=8)
axes[1].set_title('(b) IVFPQ', fontsize=DASH_TITLE_FS, fontweight='bold', pad=12)
axes[1].legend(loc='lower right', ncol=2, fontsize=10, frameon=True)

if not df_ivfpq_plain_dashboard.empty:
    best_i = df_ivfpq_plain_dashboard.loc[df_ivfpq_plain_dashboard['mean_accuracy'].idxmax()]
    axes[1].scatter([best_i['search_param_value']], [best_i['mean_accuracy']],
                    s=220, marker='*', color='black', zorder=6)
    axes[1].annotate(f"Best: {best_i['mean_accuracy']*100:.2f}%",
                     (best_i['search_param_value'], best_i['mean_accuracy']),
                     xytext=(-10,8), textcoords='offset points', fontsize=12, fontweight='bold')

# Panel 3: OIVFPQ
for i, config in enumerate(large_configs):
    subset = df_oivfpq[df_oivfpq['config_label'] == config]
    axes[2].plot(subset['search_param_value'], subset['mean_accuracy'],
                 marker=hnsw_markers[i % len(hnsw_markers)],
                 markersize=9, linewidth=2.2,
                 color=large_colors.get(config, '#111111'),
                 label=config)
axes[2].set_xscale('log', base=2)
axes[2].set_xticks([512, 1024, 2048, 4096, 8192, 16384, 32768])
axes[2].set_xticklabels(['512', '1k', '2k', '4k', '8k', '16k', '32k'], fontsize=16)
axes[2].tick_params(axis='y', labelsize=16)
axes[2].set_xlabel('nprobe Parameter', fontsize=DASH_LABEL_FS, fontweight='bold', labelpad=8)
axes[2].set_ylabel('Candidate Fidelity', fontsize=DASH_LABEL_FS, fontweight='bold', labelpad=8)
axes[2].set_title('(c) O-IVFPQ', fontsize=DASH_TITLE_FS, fontweight='bold', pad=12)
axes[2].legend(loc='lower right', fontsize=11, frameon=True)

if not df_oivfpq.empty:
    best_o = df_oivfpq.loc[df_oivfpq['mean_accuracy'].idxmax()]
    axes[2].scatter([best_o['search_param_value']], [best_o['mean_accuracy']],
                    s=220, marker='*', color='black', zorder=6)
    axes[2].annotate(f"Best: {best_o['mean_accuracy']*100:.2f}%",
                     (best_o['search_param_value'], best_o['mean_accuracy']),
                     xytext=(8,8), textcoords='offset points', fontsize=12)

plt.tight_layout()

png_path = os.path.join(OUT_DIR, 'accuracy_scaling_dashboard_best.png')
pdf_path = os.path.join(OUT_DIR, 'accuracy_scaling_dashboard_best.pdf')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, bbox_inches='tight', format='pdf')
plt.show()
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")

---
## 7. System Latency (Median)

Bar chart showing system latency (seconds) for different configurations.

In [ ]:
# ============================================================
# 7. System Latency (Median)
# ============================================================
import matplotlib.patheffects as path_effects
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
import matplotlib.transforms as mtransforms

def clean_val(x):
    if pd.isna(x): return np.nan
    s = str(x)
    try:
        if ' mJ' in s:
            return float(s.replace(' mJ', ''))
        return float(s)
    except:
        return np.nan

def format_model_name(name):
    model_map = {'clip': 'CLIP', 'uniir': 'UniIR', 'flava': 'FLAVA', 'openclip': 'OpenCLIP'}
    if '+' in name:
        parts = name.split('+')
        return f"({model_map.get(parts[0], parts[0])}, {model_map.get(parts[1], parts[1])})"
    return model_map.get(name, name.upper())

csv_path = r'C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Data\master_pipeline_medians.csv'
prefix = 'median'

df_csv = pd.read_csv(csv_path)
df_single = df_csv[df_csv['pipeline'] == 'single'].copy()
df_rerank = df_csv[df_csv['pipeline'] == 'rerank'].copy()
df_rrf = df_csv[df_csv['pipeline'] == 'ensemble'].copy()
    
single_map = {}
for _, row in df_single.iterrows():
    single_map[(row['model_group'], row['index'])] = {
        'retrieval': clean_val(row[f'{prefix}_retrieval_time']),
        'encoding': clean_val(row[f'{prefix}_embedding_generation_time'])
    }
    
rerank_map = {}
for _, row in df_rerank.iterrows():
    rerank_map[(row['endpoint'], row['model_group'], row['index'])] = {
        'total': clean_val(row[f'{prefix}_total_time']),
        'retrieval': clean_val(row[f'{prefix}_retrieval_time']),
        'encoding': clean_val(row[f'{prefix}_embedding_generation_time'])
    }

data_lat = []
models_ord = ['clip', 'uniir', 'flava', 'openclip']
endpoints_ord = ['retrieve_and_rerank_blip2', 'retrieve_and_rerank_qwen']
indices = ['hnsw', 'ivfpq', 'knn']

for m in models_ord:
    for ep in endpoints_ord:
        for idx in indices:
            s_key = (m, idx)
            r_key = (ep, m, idx)
            s_data = single_map.get(s_key, {'retrieval': float('nan'), 'encoding': float('nan')})
            r_data = rerank_map.get(r_key, {'total': float('nan'), 'retrieval': float('nan'), 'encoding': float('nan')})
            ep_label = r' $\rightarrow$ BLIP2' if 'blip2' in ep else r' $\rightarrow$ Qwen'
            
            data_lat.append({
                'Model': format_model_name(m) + ep_label,
                'Index': 'HNSW' if idx=='hnsw' else ('IVFPQ' if idx=='ivfpq' else 'Flat Index'),
                'RerankTime': r_data['total'],
                'RetrievalTime': s_data['retrieval'],
                'EncodingTime': s_data['encoding']
            })

for _, row in df_rrf.iterrows():
    m = row['model_group']
    idx = row['index']
    idx_up = 'HNSW' if idx=='hnsw' else ('IVFPQ' if idx=='ivfpq' else 'Flat Index')
    
    emb_t = clean_val(row[f'{prefix}_embedding_generation_time'])
    ret_t = clean_val(row[f'{prefix}_retrieval_time'])
    fusion_t = clean_val(row[f'{prefix}_rrf_fusion_time'])
    
    tot_t = row[f'{prefix}_total_time']
    if pd.isna(tot_t):
        tot_t = (emb_t if not pd.isna(emb_t) else 0) + \
                (ret_t if not pd.isna(ret_t) else 0) + \
                (fusion_t if not pd.isna(fusion_t) else 0)
                
    if pd.isna(tot_t): tot_t = 0
    
    data_lat.append({
        'Model': format_model_name(m),
        'Index': idx_up,
        'RerankTime': tot_t,
        'RetrievalTime': tot_t,
        'EncodingTime': 0
    })

df = pd.DataFrame(data_lat)
df['EncodingTime'] = df['EncodingTime'].fillna(0)
df['RetrievalTime'] = df['RetrievalTime'].fillna(0)
df['VanillaTotal'] = df['RetrievalTime'] + df['EncodingTime']

plt.figure(figsize=(26, 12))

colors = {'HNSW': '#3B82F6', 'IVFPQ': '#8B5CF6', 'Flat Index': '#EC4899'}
palette = [colors['HNSW'], colors['IVFPQ'], colors['Flat Index']]
hue_order = ['HNSW', 'IVFPQ', 'Flat Index']

ax = sns.barplot(
    data=df, x='Model', y='RerankTime', hue='Index',
    hue_order=hue_order, palette=palette,
    alpha=0.3, edgecolor='black', linewidth=1.5,
    err_kws={'linewidth': 0}
)

sns.barplot(
    data=df, x='Model', y='VanillaTotal', hue='Index',
    hue_order=hue_order, palette=palette,
    alpha=1.0, edgecolor='black', linewidth=1.5,
    ax=ax,
    err_kws={'linewidth': 0}
)

sns.barplot(
    data=df, x='Model', y='EncodingTime', hue='Index',
    hue_order=hue_order, palette=palette,
    alpha=1.0, edgecolor='black', linewidth=1.5,
    hatch='//',
    ax=ax,
    err_kws={'linewidth': 0}
)

ax.set_yscale("log")
ax.set_xlabel('Model', fontsize=42, fontweight='bold')
ax.set_ylabel('Latency (s)', fontsize=42, fontweight='bold')

ax.tick_params(axis='y', labelsize=28)
ax.set_xticks(range(len(df['Model'].unique())))
ax.set_xticklabels(df['Model'].unique(), rotation=45, ha='right', fontsize=33)

ax.grid(axis='y', which='major', linestyle='-', alpha=0.5)
ax.grid(axis='y', which='minor', linestyle=':', alpha=0.3)

light_patches = [Patch(facecolor=colors[t], alpha=0.3, edgecolor='black') for t in hue_order]
dark_patches = [Patch(facecolor=colors[t], alpha=1.0, edgecolor='black') for t in hue_order]
hatched_patches = [Patch(facecolor=colors[t], alpha=1.0, hatch='//', edgecolor='black') for t in hue_order]

legend_handles = [
    Patch(alpha=0, label='Indices:'),
    Patch(facecolor=colors['HNSW'], edgecolor='black', label='HNSW'),
    Patch(facecolor=colors['IVFPQ'], edgecolor='black', label='IVFPQ'),
    Patch(facecolor=colors['Flat Index'], edgecolor='black', label='Flat Index'),
    Patch(alpha=0, label='Pipeline Stages:'),
    tuple(light_patches),
    tuple(dark_patches),
    tuple(hatched_patches)
]

labels = [h.get_label() if hasattr(h, 'get_label') else '' for h in legend_handles]
labels[5] = 'Retrieve-and-rerank'
labels[6] = 'Single Index Retrieval'
labels[7] = 'Encoding Time'

ax.legend(handles=legend_handles, labels=labels, 
          fontsize=32, loc='upper right', frameon=True, shadow=True,
          handler_map={tuple: HandlerTuple(ndivide=None)}, ncol=2)

patches = ax.patches
total_patches = len(patches)
chunk_size = total_patches // 3

for i in range(chunk_size):
    p_total = patches[i]
    h_total = p_total.get_height()
    if pd.isna(h_total) or h_total <= 0: continue
    
    p_vanilla = patches[i + chunk_size]
    h_vanilla = p_vanilla.get_height()
    if pd.isna(h_vanilla): h_vanilla = 0
    
    if (h_total - h_vanilla) > 0.01:
         val_str = f"{h_total:.2f}"
         ax.annotate(val_str, 
                (p_total.get_x() + p_total.get_width() / 2., h_total), 
                ha='center', va='top', 
                xytext=(0, -8), textcoords='offset points',
                fontsize=20, fontweight='bold', color='black', rotation=90,
                path_effects=[path_effects.withStroke(linewidth=4, foreground="white")])

for i in range(chunk_size, 2*chunk_size):
    p = patches[i]
    height = p.get_height()
    if pd.isna(height) or height <= 0: continue
    
    if height < 0.01: val_str = f"{height:.4f}"
    elif height < 0.1: val_str = f"{height:.3f}"
    elif height > 100: val_str = f"{int(height)}"
    else: val_str = f"{height:.2f}"
        
    ax.annotate(val_str, 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='bottom', 
                xytext=(0, 4), textcoords='offset points',
                fontsize=18, fontweight='bold', color='black', rotation=90,
                path_effects=[path_effects.withStroke(linewidth=4, foreground="white")])
                
for i in range(2*chunk_size, 3*chunk_size):
    p_enc = patches[i]
    h_enc = p_enc.get_height()
    if pd.isna(h_enc) or h_enc <= 0: continue
    
    p_vanilla = patches[i - chunk_size]
    h_vanilla = p_vanilla.get_height()
    if pd.isna(h_vanilla): h_vanilla = 0
    
    if (h_vanilla - h_enc) > 0.005:
         if h_enc < 0.01: val_str_enc = f"{h_enc:.4f}"
         elif h_enc < 0.1: val_str_enc = f"{h_enc:.3f}"
         elif h_enc > 100: val_str_enc = f"{int(h_enc)}"
         else: val_str_enc = f"{h_enc:.2f}"
             
         ax.annotate(val_str_enc, 
                (p_enc.get_x() + p_enc.get_width() / 2., h_enc), 
                ha='center', va='bottom', 
                xytext=(0, 4), textcoords='offset points',
                fontsize=18, fontweight='bold', color='black', rotation=90,
                path_effects=[path_effects.withStroke(linewidth=4, foreground="white")])

ax.axvline(x=7.5, color='black', linestyle='--', linewidth=3.0, alpha=0.9)

def draw_curly_brace(ax, x1, x2, y_base, H, color='black', lw=2):
    trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    W = x2 - x1
    Xc = (x1 + x2) / 2.0
    num_pts = 50
    t = np.linspace(0, 1, num_pts)
    
    def bezier(p0, p1, p2, p3):
        x = (1-t)**3 * p0[0] + 3*(1-t)**2 * t * p1[0] + 3*(1-t) * t**2 * p2[0] + t**3 * p3[0]
        y = (1-t)**3 * p0[1] + 3*(1-t)**2 * t * p1[1] + 3*(1-t) * t**2 * p2[1] + t**3 * p3[1]
        return x, y
        
    s1_x, s1_y = bezier((x1, y_base), (x1, y_base - H/2.0), (x1 + W/8.0, y_base - H/2.0), (x1 + W/4.0, y_base - H/2.0))
    s2_x, s2_y = bezier((x1 + W/4.0, y_base - H/2.0), (x1 + W/3.0, y_base - H/2.0), (Xc - W/16.0, y_base - H + H/10.0), (Xc, y_base - H))
    s3_x, s3_y = bezier((Xc, y_base - H), (Xc + W/16.0, y_base - H + H/10.0), (x2 - W/3.0, y_base - H/2.0), (x2 - W/4.0, y_base - H/2.0))
    s4_x, s4_y = bezier((x2 - W/4.0, y_base - H/2.0), (x2 - W/8.0, y_base - H/2.0), (x2, y_base - H/2.0), (x2, y_base))
    
    brace_x = np.concatenate([s1_x, s2_x, s3_x, s4_x])
    brace_y = np.concatenate([s1_y, s2_y, s3_y, s4_y])
    ax.plot(brace_x, brace_y, transform=trans, color=color, lw=lw, clip_on=False)

left_lim = -0.45
right_lim = 13.20
ax.set_xlim(left_lim, right_lim)

draw_curly_brace(ax, left_lim, 7.5, 1.02, -0.05, lw=2.5)
draw_curly_brace(ax, 7.5, right_lim, 1.02, -0.05, lw=2.5)

trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
peak_left = (left_lim + 7.5) / 2.0
peak_right = (7.5 + right_lim) / 2.0
ax.text(peak_left, 1.09, 'Retrieve $\\rightarrow$ rerank', transform=trans, ha='center', va='bottom', fontsize=33, fontweight='bold')
ax.text(peak_right, 1.09, 'Ensemble', transform=trans, ha='center', va='bottom', fontsize=33, fontweight='bold')

plt.tight_layout()
plt.subplots_adjust(top=0.88)

out_file = os.path.join(OUT_DIR, 'latency_combined_rrf_median_final_braced_v3.png')
pdf_file = os.path.join(OUT_DIR, 'latency_combined_rrf_median_final_braced_v3.pdf')
plt.savefig(out_file, dpi=300)
plt.savefig(pdf_file, format='pdf')
plt.show()
print(f"Saved: {out_file}")
print(f"Saved: {pdf_file}")


---
## 8. System Energy

Bar chart showing system energy (mJ) for different configurations.

In [ ]:
# ============================================================
# 8. System Energy
# ============================================================

df_csv = pd.read_csv(csv_path)

df_single = df_csv[df_csv['pipeline'] == 'single'].copy()
df_rerank = df_csv[df_csv['pipeline'] == 'rerank'].copy()
df_rrf = df_csv[df_csv['pipeline'] == 'ensemble'].copy()

single_map = {}
for _, row in df_single.iterrows():
    single_map[(row['model_group'], row['index'])] = clean_val(row['batch_energy_total_mj']) 
    
rerank_map = {}
for _, row in df_rerank.iterrows():
    rerank_map[(row['endpoint'], row['model_group'], row['index'])] = clean_val(row['batch_energy_total_mj'])

data_eng = []
models_ord = ['clip', 'uniir', 'flava', 'openclip']
endpoints_ord = ['retrieve_and_rerank_blip2', 'retrieve_and_rerank_qwen']
indices = ['hnsw', 'ivfpq', 'knn']

for m in models_ord:
    for ep in endpoints_ord:
        for idx in indices:
            s_key = (m, idx)
            r_key = (ep, m, idx)
            energy_single = single_map.get(s_key, float('nan'))
            energy_rerank = rerank_map.get(r_key, float('nan'))
            
            ep_label = r' $\rightarrow$ BLIP2' if 'blip2' in ep else r' $\rightarrow$ Qwen'
            
            data_eng.append({
                'Model': format_model_name(m) + ep_label,
                'Index': 'HNSW' if idx=='hnsw' else ('IVFPQ' if idx=='ivfpq' else 'Flat Index'),
                'Total Energy': energy_rerank,
                'Retrieval Energy': energy_single
            })

for _, row in df_rrf.iterrows():
    m = row['model_group']
    idx = row['index']
    idx_up = 'HNSW' if idx=='hnsw' else ('IVFPQ' if idx=='ivfpq' else 'Flat Index')
    
    energy_rrf = clean_val(row['batch_energy_total_mj'])
    if pd.isna(energy_rrf): energy_rrf = float('nan')
    
    data_eng.append({
        'Model': format_model_name(m),
        'Index': idx_up,
        'Total Energy': energy_rrf,
        'Retrieval Energy': energy_rrf
    })

df_final = pd.DataFrame(data_eng)

plt.figure(figsize=(26, 12))

colors = {'HNSW': '#3B82F6', 'IVFPQ': '#8B5CF6', 'Flat Index': '#EC4899'}
palette = [colors['HNSW'], colors['IVFPQ'], colors['Flat Index']]
hue_order = ['HNSW', 'IVFPQ', 'Flat Index']

ax = sns.barplot(
    data=df_final, x='Model', y='Total Energy', hue='Index',
    hue_order=hue_order, palette=palette,
    alpha=0.3, edgecolor='black', linewidth=1.5,
    err_kws={'linewidth': 0}
)

sns.barplot(
    data=df_final, x='Model', y='Retrieval Energy', hue='Index',
    hue_order=hue_order, palette=palette,
    alpha=1.0, edgecolor='black', linewidth=1.5,
    ax=ax,
    err_kws={'linewidth': 0}
)

ax.set_yscale("log")
ax.set_ylabel('Energy (mJ)', fontsize=42, fontweight='bold')
ax.set_xlabel('Model', fontsize=42, fontweight='bold')

ax.tick_params(axis='y', labelsize=28)
ax.set_xticks(range(len(df_final['Model'].unique())))
ax.set_xticklabels(df_final['Model'].unique(), rotation=45, ha='right', fontsize=33)

ax.grid(axis='y', which='major', linestyle='-', alpha=0.5)
ax.grid(axis='y', which='minor', linestyle=':', alpha=0.3)

light_patches = [Patch(facecolor=colors[t], alpha=0.3, edgecolor='black') for t in hue_order]
dark_patches = [Patch(facecolor=colors[t], alpha=1.0, edgecolor='black') for t in hue_order]

legend_handles = [
    Patch(alpha=0, label='Indices:'),
    Patch(facecolor=colors['HNSW'], edgecolor='black', label='HNSW'),
    Patch(facecolor=colors['IVFPQ'], edgecolor='black', label='IVFPQ'),
    Patch(facecolor=colors['Flat Index'], edgecolor='black', label='Flat Index'),
    Patch(alpha=0, label='Pipeline Stages:'),
    tuple(light_patches),
    tuple(dark_patches),
    Patch(alpha=0, label='')
]

labels = [h.get_label() if hasattr(h, 'get_label') else '' for h in legend_handles]
labels[5] = 'Retrieve-and-rerank'
labels[6] = 'Single Index Retrieval'

ax.legend(handles=legend_handles, labels=labels, 
          fontsize=32, loc='upper right', frameon=True, shadow=True,
          handler_map={tuple: HandlerTuple(ndivide=None)}, ncol=2)

patches = ax.patches
mid_point = len(patches) // 2

for i in range(mid_point):
    p = patches[i]
    height = p.get_height()
    if pd.isna(height) or height <= 0: continue
    
    inner_h = patches[i+mid_point].get_height()
    is_rrf = False
    if not pd.isna(inner_h) and abs(inner_h - height) < 0.0001:
        is_rrf = True
        
    if not is_rrf:
         if height > 1e6: val_str = f"{height/1e6:.1f}M"
         elif height > 1e3: val_str = f"{height/1e3:.1f}k"
         else: val_str = f"{int(height)}"
             
         ax.annotate(val_str, 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='top', 
                xytext=(0, -8), textcoords='offset points',
                fontsize=20, fontweight='bold', color='black', rotation=90,
                path_effects=[path_effects.withStroke(linewidth=4, foreground="white")])

n_models = len(df_final['Model'].unique())

for i in range(mid_point, len(patches)):
    p = patches[i]
    height = p.get_height()
    if pd.isna(height) or height <= 0: continue
    
    k = i - mid_point
    group_idx = k // n_models
    
    rot = 90
    
    if height > 1e6: val_str = f"{height/1e6:.1f}M"
    elif height > 1e3: val_str = f"{height/1e3:.1f}k"
    else: val_str = f"{int(height)}"
        
    ax.annotate(val_str, 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='bottom', 
                xytext=(0, 4), textcoords='offset points',
                fontsize=18, fontweight='bold', color='black', rotation=rot,
                path_effects=[path_effects.withStroke(linewidth=4, foreground="white")])

unique_models = df_final['Model'].unique().tolist()
n_models_x = len(unique_models)

for i in range(n_models_x - 1):
    boundary = i + 0.5
    if i == 7:
        ax.axvline(x=boundary, color='black', linewidth=3, linestyle='--')
    else:
        ax.axvline(x=boundary, color='gray', linewidth=1, linestyle='-', alpha=0.5)

left_lim = -0.45
right_lim = 13.20
ax.set_xlim(left_lim, right_lim)

draw_curly_brace(ax, left_lim, 7.5, 1.02, -0.05, lw=2.5)
draw_curly_brace(ax, 7.5, right_lim, 1.02, -0.05, lw=2.5)

trans = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
peak_left = (left_lim + 7.5) / 2.0
peak_right = (7.5 + right_lim) / 2.0
ax.text(peak_left, 1.09, 'Retrieve $\\rightarrow$ rerank', transform=trans, ha='center', va='bottom', fontsize=33, fontweight='bold')
ax.text(peak_right, 1.09, 'Ensemble', transform=trans, ha='center', va='bottom', fontsize=33, fontweight='bold')

plt.tight_layout()
plt.subplots_adjust(top=0.88)

out_file = os.path.join(OUT_DIR, 'energy_combined_final_braced_v3.png')
pdf_file = os.path.join(OUT_DIR, 'energy_combined_final_braced_v3.pdf')
plt.savefig(out_file, dpi=300)
plt.savefig(pdf_file, format='pdf')
plt.show()
print(f"Saved: {out_file}")
print(f"Saved: {pdf_file}")


---
## 9. Recall@k (No Text Models) - COCO

Line chart showing recall@k curve for COCO dataset.

In [ ]:
# ============================================================
# 9. Recall@k (No Text Models) - COCO
# ============================================================

csv_path = r'C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Data\recall_at_k_no_text.csv'
df_all = pd.read_csv(csv_path)
df_coco = df_all[df_all['dataset'] == 'coco']

plt.figure(figsize=(26, 16))

for model_label, model_df in df_coco.groupby('model_label'):
    model_df = model_df.sort_values(by='k')
    plt.plot(model_df['k'], model_df['recall'], label=model_label, linestyle='-', linewidth=4)
    
xlabel_fs = 85 
ylabel_fs = 85 
ticks_fs = 67 
legend_fs = 50 

plt.xlabel('k', fontsize=xlabel_fs, fontweight='bold')
plt.ylabel('Recall @ k', fontsize=ylabel_fs, fontweight='bold')

plt.xticks(fontsize=ticks_fs)
plt.yticks(fontsize=ticks_fs)
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend(loc='lower right', fontsize=legend_fs)

plt.tight_layout()

out_file = os.path.join(OUT_DIR, 'coco_recall_at_k_no_text.png')
pdf_file = os.path.join(OUT_DIR, 'coco_recall_at_k_no_text.pdf')
plt.savefig(out_file, dpi=300)
plt.savefig(pdf_file, format='pdf')
plt.show()
print(f"Saved: {out_file}")
print(f"Saved: {pdf_file}")


---
## 10. Recall@k (No Text Models) - Flickr

Line chart showing recall@k curve for Flickr dataset.

In [ ]:
# ============================================================
# 10. Recall@k (No Text Models) - Flickr
# ============================================================

df_flickr = df_all[df_all['dataset'] == 'flickr']

plt.figure(figsize=(26, 16))

for model_label, model_df in df_flickr.groupby('model_label'):
    model_df = model_df.sort_values(by='k')
    plt.plot(model_df['k'], model_df['recall'], label=model_label, linestyle='-', linewidth=4)
    
xlabel_fs = 85 
ylabel_fs = 85 
ticks_fs = 67 
legend_fs = 50 

plt.xlabel('k', fontsize=xlabel_fs, fontweight='bold')
plt.ylabel('Recall @ k', fontsize=ylabel_fs, fontweight='bold')

plt.xticks(fontsize=ticks_fs)
plt.yticks(fontsize=ticks_fs)
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend(loc='lower right', fontsize=legend_fs)

plt.tight_layout()

out_file = os.path.join(OUT_DIR, 'flickr_recall_at_k_no_text.png')
pdf_file = os.path.join(OUT_DIR, 'flickr_recall_at_k_no_text.pdf')
plt.savefig(out_file, dpi=300)
plt.savefig(pdf_file, format='pdf')
plt.show()
print(f"Saved: {out_file}")
print(f"Saved: {pdf_file}")


---
## 11. Upset Plots (Potential vs Actual RRF) - COCO

Dual-axis bar charts for R@1, R@5, R@10 showing the potential gain vs actual RRF score for MS-COCO.

In [ ]:
# ============================================================
# 11. Upset Plots (Potential vs Actual RRF) - COCO
# ============================================================

def format_model_name(raw_name):
    name_map = {'flava': 'FLAVA', 'uniir': 'UniIR', 'clip': 'CLIP', 'minilm': 'MiniLM', 'openclip': 'OpenCLIP'}
    modality_map = {'text': 'Text', 'image': 'Image', 'joint-image-text': 'Joint', 'joint': 'Joint'}
    
    parts = raw_name.split('_', 1)
    if len(parts) == 2:
        model, mod = parts[0], parts[1]
    else:
        model, mod = raw_name, ""
        
    display_model = name_map.get(model, model.title())
    display_mod = modality_map.get(mod, mod.title())
    
    if mod:
        return f"$\\mathrm{{{display_model}}}_{{ \\mathrm{{{display_mod}}} }}$"
    return display_model

def create_stacked_potential_plot(df, metric, dataset_name, total_queries):
    df_metric = df[df['Metric'] == metric].copy()
    exclude_models = ["uniir_text", "flava_text", "openclip_text", "minilm"]
    for ex in exclude_models:
        df_metric = df_metric[~df_metric['Combo'].str.contains(ex)]
        
    df_metric = df_metric.sort_values(by='RRF Score', ascending=False).head(15)
    
    combos = df_metric['Combo']
    intersection_cnt = df_metric['Intersection']
    unique_m1_cnt = df_metric['Unique M1']
    unique_m2_cnt = df_metric['Unique M2']
    rrf_score_pct = (df_metric['RRF Score'] / total_queries) * 100
    
    clean_labels = []
    for _, row in df_metric.iterrows():
        m1_fmt = format_model_name(row['Model 1'])
        m2_fmt = format_model_name(row['Model 2'])
        clean_labels.append(f"{m1_fmt} +\n{m2_fmt}")
        
    fig, ax1 = plt.subplots(figsize=(36, 24))
    x_pos = range(len(combos))
    bar_width = 0.6
    
    p1 = ax1.bar(x_pos, intersection_cnt, width=bar_width, label='Intersection (Both Correct)', color='#dcdcdc', alpha=0.9, edgecolor='white')
    p2 = ax1.bar(x_pos, unique_m1_cnt, bottom=intersection_cnt, width=bar_width, label='Unique to Model 1', color='#4C72B0', alpha=0.9, edgecolor='white')
    p3 = ax1.bar(x_pos, unique_m2_cnt, bottom=intersection_cnt + unique_m1_cnt, width=bar_width, label='Unique to Model 2', color='#DD8452', alpha=0.9, edgecolor='white')
    
    ax1.set_xlim(-0.6, len(combos) - 0.4)
    ax1.set_ylabel('Number of Queries', fontsize=60, fontweight='bold', labelpad=30)
    ax1.set_xlabel('Model Combinations', fontsize=60, fontweight='bold', labelpad=40)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(clean_labels, rotation=90, ha='center', va='top', fontsize=42, fontweight='bold')
    ax1.tick_params(axis='y', labelsize=42)
    
    ax2 = ax1.twinx()
    star = ax2.scatter(x_pos, rrf_score_pct, marker='*', s=600, color='black', zorder=10, label='Actual RRF Recall (%)')
    
    for i, score in enumerate(rrf_score_pct):
        ax2.text(i, score + 1.2, f"{score:.2f}%", ha='center', va='bottom', fontsize=32, fontweight='bold', color='black')
        
    ax2.set_ylabel(f'Recall@{metric.replace("R", "")} (%)', fontsize=60, fontweight='bold', rotation=270, labelpad=50)
    ax2.tick_params(axis='y', labelsize=42)
    
    # Calculate adaptive limits
    max_val = max((intersection_cnt + unique_m1_cnt + unique_m2_cnt).max(), df_metric['RRF Score'].max())
    min_val = min(intersection_cnt.min(), df_metric['RRF Score'].min())
    
    # Add buffer
    range_val = max_val - min_val
    max_ylim_count = int(max_val + 0.1 * range_val)
    # Ensure nice round number for max limit if possible
    max_ylim_count = (max_ylim_count // 100 + 1) * 100
    
    min_ylim_count = max(0, int(min_val - 0.2 * range_val))
    min_ylim_count = (min_ylim_count // 100) * 100
    
    # Fallbacks based on original script if bounds don't make sense
    if dataset_name == 'coco':
        if max_ylim_count < 4700: max_ylim_count = 4700
        if min_ylim_count > 2500: min_ylim_count = 2500
    elif dataset_name == 'flickr':
        if max_ylim_count < 1000: max_ylim_count = 1000
        if min_ylim_count > 650: min_ylim_count = 650

    max_ylim_pct = (max_ylim_count / total_queries) * 100
    min_ylim_pct = (min_ylim_count / total_queries) * 100
    
    ax1.set_ylim(min_ylim_count, max_ylim_count)
    ax2.set_ylim(min_ylim_pct, max_ylim_pct)
    
    # Create 5 ticks roughly
    step = (max_ylim_count - min_ylim_count) // 4
    if step == 0: step = 100
    y_ticks = list(range(min_ylim_count, max_ylim_count + step, step))
    if y_ticks[-1] != max_ylim_count: y_ticks[-1] = max_ylim_count
    
    ax1.set_yticks(y_ticks)
    ax2.set_yticks([(t / total_queries) * 100 for t in y_ticks])
    
    handles = [star, p3, p2, p1]
    labels = ['Actual RRF Recall (%)', 'Unique to Model 2', 'Unique to Model 1', 'Intersection (Both Correct)']
    
    ax1.legend(handles, labels, loc='lower right', fontsize=40, framealpha=0.95)
    ax1.yaxis.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    filename_png = f"potential_vs_actual_dual_axis_{metric}_{dataset_name}.png"
    filename_pdf = f"potential_vs_actual_dual_axis_{metric}_{dataset_name}.pdf"
    output_path_png = os.path.join(OUT_DIR, filename_png)
    output_path_pdf = os.path.join(OUT_DIR, filename_pdf)
    
    plt.savefig(output_path_png, dpi=300)
    plt.savefig(output_path_pdf, dpi=300, format='pdf')
    plt.show()
    print(f"Saved: {output_path_png}")
    print(f"Saved: {output_path_pdf}")

csv_coco = r'C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Data\combo_unique_analysis_coco.csv'
df_coco = pd.read_csv(csv_coco)

for metric in ['R1', 'R5', 'R10']:
    print(f"Generating COCO plot for {metric}...")
    create_stacked_potential_plot(df_coco, metric, 'coco', total_queries=5000.0)


---
## 12. Upset Plots (Potential vs Actual RRF) - Flickr

Dual-axis bar charts for R@1, R@5, R@10 showing the potential gain vs actual RRF score for Flickr30k.

In [ ]:
# ============================================================
# 12. Upset Plots (Potential vs Actual RRF) - Flickr
# ============================================================

csv_flickr = r'C:\Users\DELL\Desktop\CSE\Plots\MmirPlots\Data\combo_unique_analysis_flickr.csv'
df_flickr = pd.read_csv(csv_flickr)

for metric in ['R1', 'R5', 'R10']:
    print(f"Generating Flickr plot for {metric}...")
    create_stacked_potential_plot(df_flickr, metric, 'flickr', total_queries=1000.0)


---
## 13. Storage Comparison Log

Bar chart showing storage sizes for different index configurations on a log scale.

In [ ]:
# ============================================================
# 13. Storage Comparison Log
# ============================================================

# Base vector storage in MB
base_mb = 253.68

# Data
data = [
    {"Algorithm": "HNSW", "Config": "m16 efc100", "Storage_GB": 20.97 + base_mb/1024},
    {"Algorithm": "HNSW", "Config": "m32 efc100", "Storage_GB": 21.80 + base_mb/1024},
    {"Algorithm": "HNSW", "Config": "m48 efc400", "Storage_GB": 22.64 + base_mb/1024},
    {"Algorithm": "HNSW", "Config": "m64 efc800", "Storage_GB": 23.47 + base_mb/1024},
    
    {"Algorithm": "IVFPQ", "Config": "m16 nlist2048", "Storage_GB": (166.98 + base_mb)/1024},
    {"Algorithm": "IVFPQ", "Config": "m32 nlist8192", "Storage_GB": (291.84 + base_mb)/1024},
    {"Algorithm": "IVFPQ", "Config": "m64 nlist8192", "Storage_GB": (505.46 + base_mb)/1024},
    {"Algorithm": "IVFPQ", "Config": "m96 nlist32768", "Storage_GB": (792.0 + base_mb)/1024},

    {"Algorithm": "O-IVFPQ", "Config": "opq192 ivf16384", "Storage_GB": 1.35 + base_mb/1024},
    {"Algorithm": "O-IVFPQ", "Config": "opq192 ivf32768", "Storage_GB": 1.40 + base_mb/1024},
    {"Algorithm": "O-IVFPQ", "Config": "opq256 ivf16384", "Storage_GB": 1.77 + base_mb/1024},
    {"Algorithm": "O-IVFPQ", "Config": "opq256 ivf32768", "Storage_GB": 1.90 + base_mb/1024},

    {"Algorithm": "KNN", "Config": "Exact", "Storage_GB": 21.0 + base_mb/1024},
]

df = pd.DataFrame(data)
df = df.rename(columns={"Algorithm": "Indexing Method"})

# Plot 2: Log Scale Bar Plot
plt.figure(figsize=(14, 8))

ax = sns.barplot(data=df, x="Config", y="Storage_GB", hue="Indexing Method", dodge=False)
ax.set_yscale("log")
plt.xticks(rotation=45, ha="right", fontsize=24)
plt.yticks(fontsize=24)
plt.ylabel("Storage (GB) - Log Scale", fontsize=30, weight='bold')
plt.xlabel("Configuration", fontsize=30, weight='bold')

# Annotate bars with exact values
for p in ax.patches:
    height = p.get_height()
    if not pd.isna(height) and height > 0:
        if height > 5:
            ax.annotate(f'{height:.2f} GB', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='top', 
                        fontsize=18, color='black', xytext=(0, -5), 
                        textcoords='offset points', rotation=90, weight='bold')
        else:
            ax.annotate(f'{height:.2f} GB', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='bottom', 
                        fontsize=18, color='black', xytext=(0, 5), 
                        textcoords='offset points', rotation=90, weight='bold')

plt.legend(title="Indexing Method", loc='upper center', ncol=4, fontsize=16, title_fontsize=16)
plt.tight_layout()

png_path = os.path.join(OUT_DIR, "storage_comparison_log.png")
pdf_path = os.path.join(OUT_DIR, "storage_comparison_log.pdf")
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(pdf_path, bbox_inches='tight')
plt.show()
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")
